# Your First LLM Call

> **Goal:** Make a working API call in under 2 minutes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook gets you from zero to working LLM call as fast as possible.

## Setup (30 seconds)

Run this cell to install the SDK:

In [ ]:
# Install the Anthropic SDK
!pip install -q anthropic

## Your First Call (60 seconds)

Replace `your-api-key` with your actual API key, then run:

In [ ]:
import anthropic

# Initialize the client
client = anthropic.Anthropic(
    api_key="your-api-key"  # Or set ANTHROPIC_API_KEY environment variable
)

# Make your first call
message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "What is the capital of France? Reply in one sentence."}
    ]
)

print(message.content[0].text)

**That's it!** You just made your first LLM API call.

---

## Understanding What Happened

Let's break down the call:

In [ ]:
# The response object contains more than just text
print("=== Response Details ===")
print(f"Model: {message.model}")
print(f"Stop reason: {message.stop_reason}")
print(f"Input tokens: {message.usage.input_tokens}")
print(f"Output tokens: {message.usage.output_tokens}")
print(f"\nContent: {message.content[0].text}")

### Key Concepts

| Component | What It Does |
|-----------|-------------|
| `model` | Which LLM to use (different capabilities/costs) |
| `max_tokens` | Maximum response length |
| `messages` | Conversation history (list of user/assistant turns) |
| `usage` | Token counts (for cost tracking) |

## Modify This: Your Turn

Try changing the prompt:

In [ ]:
# TODO: Change this prompt to ask something you're curious about
your_question = "Explain recursion in programming like I'm 10 years old."

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    messages=[{"role": "user", "content": your_question}]
)

print(response.content[0].text)

## Multi-Turn Conversations

LLMs are stateless - you maintain context by passing conversation history:

In [ ]:
# Multi-turn conversation
conversation = [
    {"role": "user", "content": "My name is Alice."},
    {"role": "assistant", "content": "Hello Alice! Nice to meet you. How can I help you today?"},
    {"role": "user", "content": "What's my name?"}  # Tests if it remembers
]

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    messages=conversation
)

print(response.content[0].text)

### Key Insight

The model doesn't actually "remember" - it reads the entire conversation every time. This is why:
- **Context windows matter** (there's a limit to how much history fits)
- **Memory systems are needed** (for longer conversations)
- **Each call is independent** (stateless API)

## System Prompts: Shaping Behavior

The system prompt sets the assistant's personality and constraints:

In [ ]:
# System prompt shapes behavior
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    system="You are a pirate. Respond to everything in pirate speak, matey!",
    messages=[{"role": "user", "content": "How do I write a for loop in Python?"}]
)

print(response.content[0].text)

## Streaming: Real-Time Responses

For better UX, stream responses token by token:

In [ ]:
# Streaming response
print("Streaming response: ", end="")

with client.messages.stream(
    model="claude-sonnet-4-20250514",
    max_tokens=256,
    messages=[{"role": "user", "content": "Count from 1 to 10 slowly."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

print()  # Newline at end

## Cost Awareness

Every call costs money. Track your usage:

In [ ]:
# Approximate cost calculation (check current pricing!)
# Claude 3.5 Sonnet: $3/MTok input, $15/MTok output (as of late 2024)

def estimate_cost(usage, input_price_per_mtok=3.0, output_price_per_mtok=15.0):
    input_cost = (usage.input_tokens / 1_000_000) * input_price_per_mtok
    output_cost = (usage.output_tokens / 1_000_000) * output_price_per_mtok
    return input_cost + output_cost

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say hello!"}]
)

cost = estimate_cost(response.usage)
print(f"Input tokens: {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")
print(f"Estimated cost: ${cost:.6f}")

## Copy-Paste Template

Here's a production-ready function you can use in your projects:

In [ ]:
import anthropic
from typing import Optional

def chat(
    prompt: str,
    system: Optional[str] = None,
    model: str = "claude-sonnet-4-20250514",
    max_tokens: int = 1024
) -> str:
    """
    Simple chat function for single-turn conversations.
    
    Args:
        prompt: Your question or request
        system: Optional system prompt to shape behavior
        model: Model to use
        max_tokens: Maximum response length
    
    Returns:
        The assistant's response text
    """
    client = anthropic.Anthropic()
    
    kwargs = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}]
    }
    
    if system:
        kwargs["system"] = system
    
    response = client.messages.create(**kwargs)
    return response.content[0].text

# Usage
result = chat("What is 2 + 2?")
print(result)

## What's Next?

You've learned the basics of LLM API calls. Next steps:

1. **[02_the_full_loop_demo.ipynb](02_the_full_loop_demo.ipynb)** - See how calls fit into a complete system
2. **[Module 03: Memory Systems](../modules/module_03_memory_systems/)** - Add RAG for knowledge
3. **[Module 04: Tool Use](../modules/module_04_tool_use/)** - Let the model take actions

---

### Key Takeaways

- LLM APIs are simple: send messages, get responses
- Models are stateless: you manage conversation history
- System prompts shape behavior
- Streaming improves user experience
- Always track token usage for cost management